In [1]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

df = pd.read_csv("nbs/other/claim_w_verified_images.csv")  # replace with your path
# df = df.fillna("")
df["unverified claim"] = df["unverified claim"].fillna("")
df["reviewed claim"] = df["reviewed claim"].fillna("")

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "image-text-to-text",
    model="/gpfs/projects/bsc14/abecerr1/hub/models--google--gemma-3-12b-it/snapshots/96b6f1eccf38110c56df3a15bffe176da04bfd80",
    device="cuda",
    torch_dtype=torch.bfloat16
)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.50, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda


In [ ]:
def create_message(row):
    unverified_claim = row["unverified claim"]
    reviewed_claim = row["reviewed claim"]
    title = row["title"]
    summary = row["summary"]
    reviewed_text = row["cr_item_reviewed_text"]
    image_path = row["image_path"]
    image_path = image_path if isinstance(image_path, str) else "nbs/other/images/no_image.jpg"
    
    txt =  f"""
    ### Analyze this example:
    
    Unverified Claim: "{unverified_claim}"
    
    Reviewed Claim: "{reviewed_claim}"
    
    Title: "{title}"
    
    Summary: "{summary}"
    
    Reviewed Text: "{reviewed_text}"
    
    An image from the fact checking source is provided.
    """
        
    return txt, image_path


In [84]:
df_neg = df[df["similarity"] == 0].sample(3, random_state=0)
df_pos = df[df["similarity"] == 1].sample(3, random_state=0)
df_sample = pd.concat([df_neg, df_pos]).sample(frac=1, random_state=0)

In [ ]:
# Instructions as a separate variable
instructions = """
Your task is to determine how likely a reviewed claim is a fact-check of an unverified claim.

Return your response as a dictionary object with:
- `relatedness_score`: float between 0 and 1 (1 means clearly a fact-check, 0 means completely unrelated)
    - 0 to 0.25: somewhat related but not a clear fact-check
    - 0.25 to 0.5: moderately related with some evidence
    - 0.5 to 0.75: highly related with strong evidence
    - 0.75 to 1.0: clearly a fact-check
    
- `justification`: concise reasoning based on text and images (if available).

Follow these reasoning steps:
1. Compare the claims for semantic similarity or direct opposition.
2. Check if the reviewed claim explicitly addresses, confirms, or debunks the unverified claim.
3. Analyze provided textual context (title, summary, article text).
4. Examine images for visual evidence related to claims (e.g., relevant text, scenes, people, manipulated content).
5. Provide a confident relatedness score.
"""
from PIL import Image

def predict_row(row):
    current_claim, img_path = create_message(row)

    messages = [
            {
                "role": "system",
                "content": [
                    {"type": "text", "text": "You are a helpful assistant specialized in matching fact-checked claims to unverified claims."}
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instructions},
                    {"type": "text", "text": current_claim},
                    {"type": "image", "url": img_path},
                    {"type": "text", "text": "Please analyze the claims, article content, and image if available. Return your answer in JSON format in a parseable way with python."}
                ]
            }
        ]


    output = pipe(text=messages, max_new_tokens=300)
    return output[0]["generated_text"][-1]["content"]


# predict_row(df_sample.iloc[0])

In [86]:
df_sample_sample.iloc[0]

Unnamed: 0.1                                                           588
Unnamed: 0                                                             607
unverified claim                         reino unido para de vacunar niños
reviewed claim           La vacunación en Reino Unido está bajo investi...
similarity                                                               0
url                             https://factual.afp.com/doc.afp.com.9X39EG
title                    La vacunación contra el covid-19 no está bajo ...
text                     La vacunación contra el covid-19 no está bajo ...
summary                  La policía británica ha confirmado que no hay ...
meta_description         Artículos en internet y publicaciones en redes...
kb_keywords              [('en internet publicaciones', 0.5776), ('inte...
meta_keywords                                                         ['']
cr_country                                                          Mexico
meta_lang                

In [88]:

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

sample_preds = []
df_sample_sample = df_sample.iloc[4:]
for idx, row in tqdm(df_sample_sample.iterrows(), total=len(df_sample_sample)):
    sample_preds.append(predict_row(row))

  0%|          | 0/2 [00:00<?, ?it/s]

nan
nbs/other/images/no_image.jpg


 50%|█████     | 1/2 [00:13<00:13, 13.17s/it]

[{'input_text': [{'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful assistant specialized in matching fact-checked claims to unverified claims.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '\nYour task is to determine how likely a reviewed claim is a fact-check of an unverified claim.\n\nReturn your response as a dictionary object with:\n- `relatedness_score`: float between 0 and 1 (1 means clearly a fact-check, 0 means completely unrelated)\n    - 0 to 0.25: somewhat related but not a clear fact-check\n    - 0.25 to 0.5: moderately related with some evidence\n    - 0.5 to 0.75: highly related with strong evidence\n    - 0.75 to 1.0: clearly a fact-check\n    \n- `justification`: concise reasoning based on text and images (if available).\n\nFollow these reasoning steps:\n1. Compare the claims for semantic similarity or direct opposition.\n2. Check if the reviewed claim explicitly addresses, confirms, or debunks the unverified claim.\n3. Analyze pro

100%|██████████| 2/2 [00:33<00:00, 16.68s/it]

[{'input_text': [{'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful assistant specialized in matching fact-checked claims to unverified claims.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '\nYour task is to determine how likely a reviewed claim is a fact-check of an unverified claim.\n\nReturn your response as a dictionary object with:\n- `relatedness_score`: float between 0 and 1 (1 means clearly a fact-check, 0 means completely unrelated)\n    - 0 to 0.25: somewhat related but not a clear fact-check\n    - 0.25 to 0.5: moderately related with some evidence\n    - 0.5 to 0.75: highly related with strong evidence\n    - 0.75 to 1.0: clearly a fact-check\n    \n- `justification`: concise reasoning based on text and images (if available).\n\nFollow these reasoning steps:\n1. Compare the claims for semantic similarity or direct opposition.\n2. Check if the reviewed claim explicitly addresses, confirms, or debunks the unverified claim.\n3. Analyze pro

In [89]:
sample_preds

['```json\n{\n  "relatedness_score": 0.95,\n  "justification": "The reviewed claim directly addresses and refutes the unverified claim by stating that vaccination in the UK is *not* under criminal investigation. The title and summary explicitly debunk the initial assertion, providing evidence from the British police and the International Criminal Court. The reviewed text is a direct counter-statement to a circulating claim, indicating a clear fact-check relationship. The image is irrelevant as it depicts a \'no image available\' placeholder."\n}\n```',
 '```json\n{\n  "relatedness_score": 0.95,\n  "justification": "The reviewed claim directly addresses the unverified claim of Bill Gates blocking the sun by explicitly denying Bill Gates\' involvement in solar geoengineering and debunking the \'chemtrails\' conspiracy theory. The title \'Smugi na niebie to chemtrails i geoinżynieria? Fake news!\' confirms that the reviewed content is a fact-check. The summary provides detailed refutation

In [2]:
import json
d_splits = json.load(open("nbs/other/splits.json", "r"))

In [ ]:
df_val = df.loc[d_splits["val"]]
df_test = df.loc[d_splits["test"]]

,Unnamed: 0.1,Unnamed: 0,unverified claim,reviewed claim,similarity,url,title,text,summary,meta_description,...,cr_image,meta_image,movies,domain,cm_authors,cr_author_name,cr_author_url,cr_item_reviewed_text,dataset,image_path
3761,4716,4877,El CEO de Pfizer no se ha vacunado contra la C...,"Pfizer CEO canceled a trip because he ""hasn't ...",0,https://www.usatoday.com/story/news/factcheck/...,"Fact check: Pfizer CEO fully vaccinated, cance...","Fact check: Pfizer CEO fully vaccinated, cance...",The CEO of Pfizer did cancel a trip to Israel ...,Pfizer CEO Albert Bourla canceled a trip to Is...,...,https://www.gannett-cdn.com/presto/2020/11/20/...,https://www.gannett-cdn.com/presto/2020/11/20/...,[],www.usatoday.com,"['Ella Lee, USA TODAY', 'Ella Lee']",USA Today,usatoday.com,"Pfizer CEO canceled a trip because he ""hasn't ...",train,NaN
1884,2386,2465,la vacunacion covid esta bajo investigacion pe...,The latest data from the UKHSA appears to show...,0,https://fullfact.org/health/january-2022-expos...,UKHSA data reaffirms that vaccines work agains...,13 January 2022\n\nFalse. No vaccine is 100% e...,The UK Health Security Agency (UKHSA) data doe...,An online article has again misinterpreted dat...,...,https://fullfact.org/media/_versions/sars-cov-...,https://fullfact.org/media/_versions/sars-cov-...,[],fullfact.org,['Full Fact'],Full Fact,NaN,The latest data from the UKHSA appears to show...,train,nbs/other/images/fullfact.org/media/_versions/...
3822,4786,4950,más de 100 países rechazaron en la ONU una res...,Eslovaquia rechazó a la OTAN,0,https://factual.afp.com/doc.afp.com.326W4GV,Eslovaquia no rechazó a la OTAN ni ha apoyado ...,Eslovaquia no rechazó a la OTAN ni ha apoyado ...,Eslovaquia no rechazó a la OTAN ni apoyó a Rus...,Un video que muestra a políticos derramando ag...,...,https://factual.afp.com/sites/default/files/st...,https://factual.afp.com/sites/default/files/st...,[],factual.afp.com,[],AFP Factual,factual.afp.com,Eslovaquia rechazó a la OTAN,train,NaN
804,1057,1090,marine le pen obtuvo 14 millones de votos,Marine Le Pen a rompu avec une tradition répub...,0,https://www.20minutes.fr/elections/presidentie...,Résultats Présidentielle 2022 : Marine Le Pen ...,« Emmanuel Macron ne fera rien pour réparer le...,Marine Le Pen a omis de féliciter Emmanuel Mac...,Battue au second tour de l’élection présidenti...,...,https://img.20mn.fr/OblN_G8jQyaH9rG4lBwO3Ck/12...,https://img.20mn.fr/OblN_G8jQyaH9rG4lBwO3Ck/12...,['https://www.ultimedia.com/deliver/generic/if...,www.20minutes.fr,"['Romarik Le Dourneuf', '20 minutes']",20 Minutes,20minutes.fr,Marine Le Pen a rompu avec une tradition répub...,train,nbs/other/images/img.20mn.fr/OblN_G8jQyaH9rG4l...
2634,3383,3502,españa minimos historicos de paro juvenil,“En este último año la tasa de paro juvenil se...,0,https://www.newtral.es/tasa-paro-juvenil-extre...,La tasa de paro juvenil en Extremadura es del ...,Cerrar Explora ...,La tasa de desempleo juvenil en Extremadura es...,La tasa de paro juvenil en Extremadura registr...,...,NaN,https://www.newtral.es/wp-content/uploads/2022...,[],www.newtral.es,"['María Pascual', 'Newtral', 'Por María Pascual']",Newtral,https://www.newtral.es/,“En este último año la tasa de paro juvenil se...,train,nbs/other/images/www.newtral.es/wp-content/upl...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3544,4406,4561,el papa lamio a un niño,Pope Francis licked a baby,1,https://factcheck.afp.com/photo-altered-show-p...,Photo altered to show Pope Francis licking baby,Photo altered to show Pope Francis licking bab...,An altered photo circulating on Reddit falsely...,"Reddit posts shared more than 10,000 times sho...",...,https://factcheck.afp.com/sites/default/files/...,https://factcheck.afp.com/sites/default/files/...,[],factcheck.afp.com,['Fact Check'],AFP Fact Check,factcheck.afp.com,Pope Francis licked a baby,train,NaN
1486,1918,1983,la ue confirma que el 5g es perjudicial para l...,The 5G radiation is bad

In [ ]:
# with open("nbs/other/llm_classifier/val_preds_raw.txt", "w") as f:
#     for idx, row in tqdm(df_val.iterrows(), total=len(df_val)):
#         f.write(predict_row(row))
#         f.write("\n")
        
# with open("nbs/other/llm_classifier/test_preds_raw.txt", "w") as f:
#     for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
#         f.write(predict_row(row))
#         f.write("\n")

In [15]:
results_val[0][-10:]

'k."\n}\n```\n'

In [48]:
import json
import re

def read_results(path):
    results_val = open(path, "r+", encoding="utf-8").read().split("```json\n")[1:]
    results_val = [res.rstrip("\n```\n") for res in results_val]


    fixed_results = []
    for res in results_val:
        try:
            parsed = json.loads(res)
            fixed_results.append(parsed)
        except json.JSONDecodeError as e:
            field = re.search(r'\"justification\": \"(.*?)\"\n', res)
            new_text = field.group(1).replace('"', "'")
            res_fixed = res.replace(field.group(1), new_text)
            parsed = json.loads(res_fixed)
            fixed_results.append(parsed)
    
    return fixed_results

eval_results = read_results("nbs/other/llm_classifier/val_preds_raw.txt")
test_results = read_results("nbs/other/llm_classifier/test_preds_raw.txt")

In [50]:
df_val_results = pd.DataFrame(eval_results)
df_test_results = pd.DataFrame(test_results)

df_val_results

,relatedness_score,justification
0,0.95,The reviewed claim directly references the Pfi...
1,0.75,The unverified claim refers to a criminal inve...
2,0.10,The unverified claim concerns countries reject...
3,0.25,The unverified claim references the number of ...
4,0.90,The reviewed claim directly quotes and then cr...
...,...,...
344,1.00,The reviewed claim is a direct translation of ...
345,0.95,The reviewed claim and unverified claim are se...
346,0.95,The reviewed claim directly states the event d...
347,1.00,"The reviewed claim directly states ""Angela Mer..."


In [53]:
d_splits = json.load(open("nbs/other/splits.json", "r"))
df_val = df.loc[d_splits["val"]]
df_test = df.loc[d_splits["test"]]

In [68]:
df_val["preds"] = df_val_results["relatedness_score"].values >= 0.5
df_test["preds"] = df_test_results["relatedness_score"].values >= 0.5

In [69]:
from sklearn.metrics import classification_report

print(classification_report(df_val["similarity"], df_val["preds"]))
print(classification_report(df_test["similarity"], df_test["preds"]))

              precision    recall  f1-score   support

           0       1.00      0.36      0.53       269
           1       0.32      1.00      0.48        80

    accuracy                           0.50       349
   macro avg       0.66      0.68      0.50       349
weighted avg       0.84      0.50      0.52       349

              precision    recall  f1-score   support

           0       1.00      0.24      0.38       298
           1       0.28      1.00      0.44        89

    accuracy                           0.41       387
   macro avg       0.64      0.62      0.41       387
weighted avg       0.83      0.41      0.40       387



LLM doesn't work well as a classifier using this prompt and methodology.